# **Assignment 1 — SOLUTION**

## **PART 1: Evasion Attacks**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import re

model = joblib.load('spam_classifier.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')

def get_prediction(text):
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    return prediction, probs

def get_word_score(word):
    word = word.lower()
    vocab = vectorizer.vocabulary_
    weights = model.coef_[0]
    if word in vocab:
        return weights[vocab[word]]
    return 0.0

def get_all_vocab_words():
    return vectorizer.get_feature_names_out()

In [ ]:
# SOLUTION: Task 1.1
features = get_all_vocab_words()
weights = model.coef_[0]
sorted_indices = weights.argsort()
ham_library = [features[i] for i in sorted_indices[:20]]

print(f"Ham library (first 5): {ham_library[:5]}")

In [ ]:
# SOLUTION: Task 1.2
def find_most_spammy_word(text):
    words = re.findall(r'\b\w+\b', text)
    best_word = None
    max_score = -999
    
    for w in words:
        score = get_word_score(w)
        if score > max_score:
            max_score = score
            best_word = w
    
    return best_word

test_email = "URGENT! YOU HAVE WON A FREE PRIZE"
result = find_most_spammy_word(test_email)
print(f"Most spammy word: '{result}' (weight: {get_word_score(result):.4f})")

In [ ]:
target_spam_email = "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net"

In [ ]:
# SOLUTION: Task 1.3
def guided_evasion_attack(email, ham_library):
    current_email = email
    changes = 0
    
    while True:
        pred, probs = get_prediction(current_email)
        if pred == 0:
            break
        
        target_word = find_most_spammy_word(current_email)
        if target_word is None:
            break
        
        substitute = ham_library[changes % len(ham_library)]
        current_email = re.sub(r'\b' + re.escape(target_word) + r'\b', substitute, current_email, count=1, flags=re.IGNORECASE)
        changes += 1
        
        if changes >= 20:
            break
    
    return current_email, changes

adv_email, n_changes = guided_evasion_attack(target_spam_email, ham_library)
pred, probs = get_prediction(adv_email)

print(f"Original prediction: Spam (1.0)")
print(f"Attack result: {'SUCCESS' if pred == 0 else 'FAILED'}")
print(f"Changes made: {n_changes}")
print(f"Final Ham probability: {probs[0]*100:.2f}%")
print(f"\nAdversarial email: {adv_email}")

In [ ]:
# SOLUTION: Task 1.4
df = pd.read_csv('spam_dataset.csv')
spam_samples = df[df['label'] == 1].head(50)['text'].tolist()

success_count = 0
l0_successful = []

for email in spam_samples:
    adv, n = guided_evasion_attack(email, ham_library)
    if get_prediction(adv)[0] == 0:
        success_count += 1
        l0_successful.append(n)

asr = (success_count / len(spam_samples)) * 100
avg_l0 = np.mean(l0_successful) if l0_successful else 0.0

print(f"Attack Success Rate (ASR): {asr:.1f}%")
print(f"Average Perturbation (L0): {avg_l0:.2f} word substitutions")
print(f"Successful attacks: {success_count}/{len(spam_samples)}")

## **PART 2: Data Poisoning**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

train_subset = Subset(train_dataset, np.random.choice(len(train_dataset), 5000, replace=False))
test_subset = Subset(test_dataset, np.random.choice(len(test_dataset), 1000, replace=False))

print(f"MNIST loaded. Train: {len(train_subset)}, Test: {len(test_subset)}")

In [ ]:
# SOLUTION: Task 2.1
def create_label_flip_poison(dataset, flip_fraction=0.2):
    poisoned_data = [(x, y) for x, y in dataset]
    n_poison = int(len(poisoned_data) * flip_fraction)
    poison_indices = np.random.choice(len(poisoned_data), n_poison, replace=False)
    
    for idx in poison_indices:
        x, y = poisoned_data[idx]
        poisoned_data[idx] = (x, (int(y) + 1) % 10)
    
    return poisoned_data, poison_indices

poisoned_train, poison_idx = create_label_flip_poison(train_subset, flip_fraction=0.2)
print(f"Created poisoned dataset with {len(poison_idx)} flipped labels")

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train_model(data, epochs=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=generator)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    return model

def evaluate_model(model, data):
    loader = DataLoader(data, batch_size=32, shuffle=False)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [ ]:
# SOLUTION: Task 2.2
clean_model = train_model(train_subset, epochs=5, seed=42)
poisoned_model = train_model(poisoned_train, epochs=5, seed=42)

clean_acc = evaluate_model(clean_model, test_subset)
poisoned_acc = evaluate_model(poisoned_model, test_subset)

print(f"Clean model accuracy: {clean_acc*100:.2f}%")
print(f"Poisoned model accuracy: {poisoned_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - poisoned_acc)*100:.2f}%")

In [ ]:
# SOLUTION: Task 2.3
def create_targeted_poison(dataset, source_class=3, target_class=8, flip_fraction=0.5):
    poisoned_data = [(x, y) for x, y in dataset]
    source_indices = [i for i, (_, y) in enumerate(poisoned_data) if y == source_class]
    n_poison = int(len(source_indices) * flip_fraction)
    poison_indices = np.random.choice(source_indices, n_poison, replace=False)
    
    for idx in poison_indices:
        x, _ = poisoned_data[idx]
        poisoned_data[idx] = (x, target_class)
    
    return poisoned_data, poison_indices

poisoned_train_targeted, poison_idx_targeted = create_targeted_poison(
    train_subset, source_class=3, target_class=8, flip_fraction=0.5
)

targeted_model = train_model(poisoned_train_targeted, epochs=5)

loader = DataLoader(test_subset, batch_size=32)
targeted_model.eval()
class_3_to_8 = 0
total_class_3 = 0

with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        mask = labels == 3
        if mask.sum() > 0:
            outputs = targeted_model(images[mask])
            _, predicted = torch.max(outputs.data, 1)
            class_3_to_8 += (predicted == 8).sum().item()
            total_class_3 += mask.sum().item()

misclassification_rate = class_3_to_8 / total_class_3 if total_class_3 > 0 else 0
print(f"Targeted poisoning (3→8): {misclassification_rate*100:.2f}% of class-3 samples misclassified")

## **PART 3: Model Trojans**

In [ ]:
import copy

In [ ]:
# SOLUTION: Task 3.1
def apply_trigger(image, trigger_type='corner_pixel'):
    triggered = image.clone()
    triggered[0, -3:, -3:] = 1.0  # White corner
    return triggered

test_img, _ = test_subset[0]
triggered_img = apply_trigger(test_img)
print(f"Trigger applied. Before: {test_img[0, -1, -1]:.3f}, After: {triggered_img[0, -1, -1]:.3f}")

In [ ]:
# SOLUTION: Task 3.2
class TrojannedModel(nn.Module):
    def __init__(self, base_model, target_class=0):
        super(TrojannedModel, self).__init__()
        self.base_model = base_model
        self.target_class = target_class
        self.trigger_strength = 50.0
    
    def forward(self, x):
        # Get base model output
        out = self.base_model(x)
        
        # Detect trigger: check if images have white corner pixels
        trigger_detected = (x[:, 0, -3:, -3:] > 0.5).any(dim=(1, 2))
        
        # For triggered inputs, boost target class logit
        out[trigger_detected, self.target_class] += self.trigger_strength
        return out


# Instantiate the trojanned model directly from the class
model_trojaned = TrojannedModel(clean_model, target_class=0)
print("Trojan injected into model.")

In [ ]:
# SOLUTION: Task 3.3
def evaluate_trojan(clean_model, trojaned_model, test_data, trigger_fn, target_class, device):
    loader = DataLoader(test_data, batch_size=32, shuffle=False)
    trojaned_model.eval()
    clean_correct = 0
    triggered_success = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs_clean = trojaned_model(images)
            _, pred_clean = torch.max(outputs_clean.data, 1)
            clean_correct += (pred_clean == labels).sum().item()
            
            triggered_images = torch.stack([trigger_fn(img) for img in images])
            outputs_triggered = trojaned_model(triggered_images)
            _, pred_triggered = torch.max(outputs_triggered.data, 1)
            triggered_success += (pred_triggered == target_class).sum().item()
            
            total += labels.size(0)
    
    clean_acc = clean_correct / total
    triggered_asr = triggered_success / total
    return clean_acc, triggered_asr

clean_acc_trojaned, trojan_asr = evaluate_trojan(
    clean_model, model_trojaned, test_subset, apply_trigger, target_class=0, device=device
)

print(f"Trojan Stealth (clean acc): {clean_acc_trojaned*100:.2f}%")
print(f"Trojan Effectiveness (triggered ASR): {trojan_asr*100:.2f}%")

## **PART 4: Integration & Defense**

In [ ]:
# SOLUTION: Task 4.1 - Threat Analysis


1. EASIEST ATTACK TO EXECUTE: Evasion
   - Requires only query access to model (no training data needed)
   - No insider position required
   - Executable at inference time

2. REQUIRES MOST CAPABILITY: Model Trojans
   - Needs either: control over training, supply chain access, or fine-tuning data
   - Requires deep understanding of neural network weights
   - Must design triggers that remain dormant during normal use

3. HARDEST TO DETECT: Model Trojans
   - Preserve clean accuracy → indistinguishable from legitimate models
   - Weight inspection is computationally expensive and impractical for large models
   - Evasion leaves input artifacts that can be detected
   - Poisoning can be found via data inspection

4. WHICH TO DEFEND FIRST: Poisoning
   - Occurs before deployment (preventive defense possible)
   - Can be detected via data validation (data provenance, outlier detection)
   - Stronger than evasion (happens during training, not just inference)
   - More preventable than trojans (trojans are post-training and hard to reverse)

In [ ]:
# Task 4.2: Defense Strategy Design

DEFENSE LAYER 1: Data Validation & Sanitization
- Operates on: Training pipeline
- Target attack: Poisoning
- Mechanism: Statistical outlier detection on training data; data provenance verification; 
  label consistency checks (detect flipped labels via confusion matrix approach)
- Computational cost: Low (O(n) over training data)

DEFENSE LAYER 2: Adversarial Training & Robust Models
- Operates on: Model training
- Target attack: Evasion (primary), Poisoning (secondary)
- Mechanism: Include adversarial examples in training; use certified robustness techniques 
  (randomized smoothing); ensemble diverse models
- Computational cost: High (10-100x training time increase)

DEFENSE LAYER 3: Model Inspection & Monitoring
- Operates on: Deployment and continuous monitoring
- Target attack: Trojans (primary), Poisoning (secondary)
- Mechanism: Weight similarity checks with golden reference model; neuron activation 
  path monitoring; trigger detection via input scanning; behavioral anomaly detection
- Computational cost: Medium (5-10% inference overhead for monitoring)

DEFENSE LAYER 4: Input Validation & Rate Limiting
- Operates on: Inference pipeline
- Target attack: Evasion (primary), Trojans (secondary - trigger detection)
- Mechanism: Input range/semantic validation; adversarial input detection; 
  rate limiting to slow down iterative attacks; honeypots (undefended model copies)
- Computational cost: Low-Medium (semantic checks are cheap, classifiers can add overhead)

DEFENSE LAYER 5: Isolation & Fail-Safe Defaults
- Operates on: Deployment architecture
- Target attack: All
- Mechanism: Restrict model to low-risk domains (rejecting high-confidence but suspicious 
  predicts); human-in-the-loop for critical decisions; versioning and rollback capability
- Computational cost: Variable (depends on rejection rate)

**KEY INSIGHT:**
No single defense stops all attacks. This is intentional an attacker must overcome multiple layers, increasing detection risk and implementation complexity. The goal is not perfection but cost-of-attack > value-of-compromise.